# Dataset Generation

In [1]:
import os
from pathlib import Path

os.chdir(Path.cwd().parents[0])
print("Now in:", Path.cwd())

gen_config_path = str(Path.cwd()) + r"\data_generation\gen_configs"

from data_processing.preprocessing_functions import create_pkt_gen_jobs, execute_pkt_gen_jobs, split_train_val_test
from data_processing.LabelLut import LABEL_OTHER_DEVICE, LABEL_DULT, STATE_LOST, STATE_NEARBY, SEPARATOR
from ble import BleStream

dataPathProcessed = str(Path.cwd()) + r"\data\csv" + r"\Processed Files\\"
dataPathPCAP = str(Path.cwd()) + r"\data\pcap" + r"\synthetic\\"

Now in: C:\Users\stsax\OneDrive\Studium\9. Semester\Masterarbeit\Repository


In [2]:
gen_jobs = create_pkt_gen_jobs({gen_config_path + r"\other_device.yaml": 250_000}, job_size=100)

df_other = execute_pkt_gen_jobs(gen_jobs)

df_other['Label'] = LABEL_OTHER_DEVICE
df_other['File'] = 'File 1'

train, val, test = split_train_val_test(df_other,
                                        file_col='File',
                                        time_col='Time',
                                        train_frac=0.8,
                                        val_frac=0.1,
                                        test_frac=0.1
                                        )

train.drop(['File'], axis=1, inplace=True)
val.drop(['File'], axis=1, inplace=True)
test.drop(['File'], axis=1, inplace=True)

train.to_parquet(dataPathProcessed + "synthetic_samples_other_device_train.parquet",
                 engine="pyarrow",
                 compression="snappy",
                 index=False)

BleStream().from_pandas(train).to_pcap_file(dataPathPCAP + "synthetic_samples_other_device_train.pcapng")

val.to_parquet(dataPathProcessed + "synthetic_samples_other_device_validation.parquet",
               engine="pyarrow",
               compression="snappy",
               index=False)

BleStream().from_pandas(val).to_pcap_file(dataPathPCAP + "synthetic_samples_other_device_validation.pcapng")

test.to_parquet(dataPathProcessed + "synthetic_samples_other_device_test.parquet",
                engine="pyarrow",
                compression="snappy",
                index=False)

BleStream().from_pandas(test).to_pcap_file(dataPathPCAP + "synthetic_samples_other_device_test.pcapng")

200000it [00:37, 5398.12it/s]
25000it [00:04, 5522.06it/s]
25000it [00:04, 5603.27it/s]


In [3]:
gen_configs_adv = [{gen_config_path + r"\adversarial_samsung_smart_things.yaml": 50_000},
                   {gen_config_path + r"\adversarial_google_find_my.yaml": 50_000},
                   {gen_config_path + r"\adversarial_tile.yaml": 50_000}
                   ]

names = ['adversarial_samsung_smart_things', 'adversarial_google_find_my', 'adversarial_tile']

for config, name in zip(gen_configs_adv, names):
    gen_jobs = create_pkt_gen_jobs(config, job_size=100)

    df_adv = execute_pkt_gen_jobs(gen_jobs)

    df_adv['Label'] = LABEL_OTHER_DEVICE
    df_adv['File'] = 'File 1'

    train, val, test = split_train_val_test(df_adv,
                                            file_col='File',
                                            time_col='Time',
                                            train_frac=0.8,
                                            val_frac=0.1,
                                            test_frac=0.1
                                            )

    train.drop(['File'], axis=1, inplace=True)
    val.drop(['File'], axis=1, inplace=True)
    test.drop(['File'], axis=1, inplace=True)

    train.to_parquet(dataPathProcessed + "synthetic_samples_" + name + "_train.parquet",
                     engine="pyarrow",
                     compression="snappy",
                     index=False)

    BleStream().from_pandas(train).to_pcap_file(dataPathPCAP + "synthetic_samples_" + name + "_train.pcapng")

    val.to_parquet(dataPathProcessed + "synthetic_samples_" + name + "_validation.parquet",
                   engine="pyarrow",
                   compression="snappy",
                   index=False)

    BleStream().from_pandas(val).to_pcap_file(dataPathPCAP + "synthetic_samples_" + name + "_validation.pcapng")

    test.to_parquet(dataPathProcessed + "synthetic_samples_" + name + "_test.parquet",
                    engine="pyarrow",
                    compression="snappy",
                    index=False)

    BleStream().from_pandas(test).to_pcap_file(dataPathPCAP + "synthetic_samples_" + name + "_test.pcapng")

40000it [00:08, 4639.15it/s]
5000it [00:01, 4685.01it/s]
5000it [00:01, 4802.12it/s]
40000it [00:08, 4672.83it/s]
5000it [00:01, 4468.22it/s]
5000it [00:01, 4589.97it/s]
40000it [00:11, 3516.02it/s]
5000it [00:01, 4144.47it/s]
5000it [00:01, 4691.61it/s]


In [4]:
gen_config_dult = {gen_config_path + r"\dult_train_nearby.yaml": 50_000}
gen_jobs = create_pkt_gen_jobs(gen_config_dult, job_size=50)

df = execute_pkt_gen_jobs(gen_jobs)

BleStream().from_pandas(df).to_pcap_file(dataPathPCAP + "synthetic_samples_dult_nearby_train.pcapng")

df['Label'] = LABEL_DULT + SEPARATOR + STATE_NEARBY

df.to_parquet(dataPathProcessed + "synthetic_samples_dult_nearby_train.parquet",
              engine="pyarrow",
              compression="snappy",
              index=False)

50000it [00:11, 4175.94it/s]


In [5]:
gen_config_dult = {gen_config_path + r"\dult_train_lost.yaml": 50_000}
gen_jobs = create_pkt_gen_jobs(gen_config_dult, job_size=50)

df = execute_pkt_gen_jobs(gen_jobs)

BleStream().from_pandas(df).to_pcap_file(dataPathPCAP + "synthetic_samples_dult_lost_train.pcapng")

df['Label'] = LABEL_DULT + SEPARATOR + STATE_LOST

df.to_parquet(dataPathProcessed + "synthetic_samples_dult_lost_train.parquet",
              engine="pyarrow",
              compression="snappy",
              index=False)

50000it [00:11, 4219.88it/s]


In [6]:
gen_config_dult = {gen_config_path + r"\dult_test_nearby.yaml": 10_000}
gen_jobs = create_pkt_gen_jobs(gen_config_dult, job_size=50)

df = execute_pkt_gen_jobs(gen_jobs)

BleStream().from_pandas(df).to_pcap_file(dataPathPCAP + "synthetic_samples_dult_nearby_test.pcapng")

df['Label'] = LABEL_DULT + SEPARATOR + STATE_NEARBY

df.to_parquet(dataPathProcessed + "synthetic_samples_dult_nearby_test.parquet",
              engine="pyarrow",
              compression="snappy",
              index=False)

10000it [00:02, 4311.46it/s]


In [7]:
gen_config_dult = {gen_config_path + r"\dult_test_lost.yaml": 10_000}
gen_jobs = create_pkt_gen_jobs(gen_config_dult, job_size=50)

df = execute_pkt_gen_jobs(gen_jobs)

BleStream().from_pandas(df).to_pcap_file(dataPathPCAP + "synthetic_samples_dult_lost_test.pcapng")

df['Label'] = LABEL_DULT + SEPARATOR + STATE_LOST

df.to_parquet(dataPathProcessed + "synthetic_samples_dult_lost_test.parquet",
              engine="pyarrow",
              compression="snappy",
              index=False)

10000it [00:02, 4388.46it/s]


In [8]:
gen_config_dult = {gen_config_path + r"\dult_train_lost.yaml": 10_000}
gen_jobs = create_pkt_gen_jobs(gen_config_dult, job_size=50)

df = execute_pkt_gen_jobs(gen_jobs)

BleStream().from_pandas(df).to_pcap_file(dataPathPCAP + "synthetic_samples_dult_lost_validation.pcapng")

df['Label'] = LABEL_DULT + SEPARATOR + STATE_LOST

df.to_parquet(dataPathProcessed + "synthetic_samples_dult_lost_validation.parquet",
              engine="pyarrow",
              compression="snappy",
              index=False)

10000it [00:02, 4320.06it/s]


In [9]:
gen_config_dult = {gen_config_path + r"\dult_train_nearby.yaml": 10_000}
gen_jobs = create_pkt_gen_jobs(gen_config_dult, job_size=50)

df = execute_pkt_gen_jobs(gen_jobs)

BleStream().from_pandas(df).to_pcap_file(dataPathPCAP + "synthetic_samples_dult_nearby_validation.pcapng")

df['Label'] = LABEL_DULT + SEPARATOR + STATE_NEARBY

df.to_parquet(dataPathProcessed + "synthetic_samples_dult_nearby_validation.parquet",
              engine="pyarrow",
              compression="snappy",
              index=False)

10000it [00:02, 3425.67it/s]
